# Семинар 9 - Методы построения оптического потока по последовательности изображений

**Этот семинар содержит оцениваемое домашнее задание**

***

Источник - https://habr.com/ru/post/201406/

$\textbf{Task statement}$: Оптический поток (ОП) – изображение видимого движения, представляющее собой сдвиг каждой точки (пикселя) между двумя изображениями.

По сути, он представляет собой поле скоростей. Суть ОП в том, что для каждой точки изображения $I_{t_0} (\vec{r})$ находится такой вектор сдвига $\delta \vec{r}$, чтобы было соответсвие между исходной точкой и точкой на следущем фрейме $I_{t_1} (\vec{r} + \delta \vec{r})$. В качестве метрики соответвия берут близость интенсивности пикселей, беря во внимание маленькую разницу по времени между кадрами: $\delta{t} = t_{1} - t_{0}$. В более точных методах точку можно привязывать к объекту на основе, например, выделения ключевых точек, а также считать градиенты вокруг точки, лапласианы и проч.

$\textbf{For what}$: Определение собственной скорости, Определение локализации, Улучшение методов трекинга объектов, сегментации, Детектирование событий, Сжатие видеопотока и проч.

![](data/tennis.png)

Разделяют 2 вида оптического потока - плотный (dense) [Farneback method, neural nets], работающий с целым изображением, и выборочный (sparse) [Lucas-Kanade method], работающий с ключевыми точками

In [ ]:
# !wget https://www.bogotobogo.com/python/OpenCV_Python/images/mean_shift_tracking/slow_traffic_small.mp4 -O data/slow_traffic_small.mp4

In [1]:
import cv2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import IPython

%matplotlib inline

## Lucas-Kanade (sparse)

Пусть $I_{1} = I(x, y, t_{1})$ интенсивность в некоторой точке (x, y) на первом изображении (т. е. в момент времени t). На втором изображении эта точка сдвинулась на (dx, dy), при этом прошло время dt, тогда $I_{2} = I(x + dx, y + dx, t_{1} + dt) \approx I_{1} + I_{x}dx + I_{y}dy +  I_{t}dt$. Из постановки задачи следует, что интенсивность пикселя не изменилась, тогда $I_{1} = I_{2}$. Далее определяем $dx, dy$.

Самое простое решение проблемы – алгоритм Лукаса-Канаде. У нас же на изображении объекты размером больше 1 пикселя, значит, скорее всего, в окрестности текущей точки у других точек будут примерно такие же сдвиги. Поэтому мы возьмем окно вокруг этой точки и минимизируем (по МНК) в нем суммарную погрешность с весовыми коэффициентами, распределенными по Гауссу, то есть так, чтобы наибольший вес имели пиксели, ближе всего находящиеся к исследуемому.

**Полезные материалы:** 
- цикл видео-лекций от First Principles of Computer Vision, посвященный Optical Flow и алгоритму Lucas-Kanade: https://youtube.com/playlist?list=PL2zRqk16wsdoYzrWStffqBAoUY8XdvatV

### Вопрос 1

Перечислите три основных предположения, на которых базируется метод Lucas-Kanade. Почему каждое из них важно для корректной работы алгоритма?

**Ответ:**

1. **Постоянство яркости (brightness constancy).** Считается, что интенсивность в точке не меняется при движении: $I(x, y, t) = I(x + u, y + v, t + 1)$. Из этого следует уравнение $I_x u + I_y v + I_t = 0$. Без этого предположения нельзя связать разность кадров $I_t$ с пространственными градиентами и оценить $(u, v)$.

2. **Малое смещение (small motion).** Смещение $(u, v)$ за один шаг времени мало, поэтому разложение Тейлора по пространству и времени корректно. При больших сдвигах линейная аппроксимация даёт большую ошибку; для этого используют пирамиду изображений и итеративное уточнение.

3. **Пространственная когерентность (локально постоянный поток).** В окне $w \times w$ все пиксели имеют одинаковый вектор $(u, v)$. Это даёт переопределённую систему уравнений и позволяет решать её методом наименьших квадратов (матрица структуры $A$ и вектор $b$).


### Вопрос 2

Объясните, зачем нужен пирамидальный подход в алгоритме Lucas-Kanade. Какую проблему он решает и как именно?

**Ответ:**

Пирамидальный LK решает проблему больших смещений, при которых обычный LK (с линейным приближением) сходится плохо или попадает в неверный локальный минимум.

Строится пирамида изображений: отслеживание начинается на верхнем уровне, где то же физическое смещение занимает меньше пикселей и удовлетворяет предположению о малом движении. Полученный вектор $(u, v)$ масштабируется (умножается на 2 при переходе на уровень выше по разрешению) и служит начальным приближением на следующем уровне, где уточняется итерациями LK. Так грубая оценка последовательно уточняется до полного разрешения.

### Вопрос 3

С какими проблемами может столкнуться алгоритм Lucas-Kanade при отслеживании точек на видео? Назовите минимум три ограничения.

**Ответ:**

1. **Окклюзии и выход точки из кадра** — объект перекрывается или уходит за границу; яркость перестаёт быть постоянной, отслеживание теряется.

3. **Изменение освещения и нарушение постоянства яркости** — тени, блики, экспозиция ломают модель $I_1 = I_2$.

3. **Однородные (мало текстурированные) области** — мало углов, $\det(A) \approx 0$, решение неустойчиво.

4. **Большие и быстрые движения** — без пирамиды и достаточного числа итераций линейное приближение не справляется.


### Задание 1

Напишите реализацию Лукаса-Канаде c помощью numpy и cv2. Сравните с реализацией `cv2.calcOpticalFlowPyrLK`.

In [2]:
def build_image_pyramid(image, num_levels, scale_factor=0.5):
    """
    Создаёт пирамиду изображений с уменьшающимся разрешением.

    Аргументы:
        image: Исходное изображение (одноканальное, grayscale)
        num_levels: Количество уровней пирамиды
        scale_factor: Коэффициент масштабирования между соседними уровнями

    Возвращает:
        Список изображений [image_level_0, image_level_1, ..., image_level_n-1], где:
        - image_level_0 - исходное изображение
        - Каждый следующий уровень уменьшен относительно предыдущего в scale_factor раз

    Примечание:
        - Используйте cv2.resize с интерполяцией cv2.INTER_LINEAR для уменьшения размера
        - Первым элементом списка должна быть копия исходного изображения
    """
    pyramid = [image.copy()]
    current = image.copy()
    for _ in range(1, num_levels):
        h, w = current.shape[:2]
        new_w = max(1, int(w * scale_factor))
        new_h = max(1, int(h * scale_factor))
        current = cv2.resize(current, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        pyramid.append(current)
    return pyramid


def compute_image_gradients(image):
    """
    Вычисляет пространственные градиенты изображения.

    Аргументы:
        image: Входное изображение (одноканальное, grayscale)

    Возвращает:
        Кортеж (Ix, Iy), где Ix и Iy - градиенты по x и y соответственно

    Примечание:
        - Используйте фильтр Собеля (cv2.Sobel) с ksize=3
        - Используйте тип данных cv2.CV_64F для более точных вычислений
    """
    Ix = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    return Ix, Iy


def compute_lk_optical_flow_point(Ix, Iy, It, window_size=5):
    """
    Вычисляет оптический поток по методу Lucas-Kanade для одного окна.

    Аргументы:
        Ix: Градиент изображения по x
        Iy: Градиент изображения по y
        It: Временной градиент (разница между кадрами)
        window_size: Размер окна для вычисления (нечетное число)

    Возвращает:
        Кортеж (u, v) компонентов вектора потока или (None, None) если решение ненадежное

    Примечание:
        - Создайте окно для градиентов, выбрав центральный пиксель и окно размером window_size x window_size
        - Вычислите сумму произведений градиентов для формирования матрицы A:
          A = [[sum(Ix*Ix), sum(Ix*Iy)], [sum(Ix*Iy), sum(Iy*Iy)]]
        - Проверьте обусловленность матрицы A через собственные значения
        - Если минимальное собственное значение меньше порога (например, 1e-4), верните (None, None)
        - Сформируйте вектор b: [-sum(Ix*It), -sum(Iy*It)]
        - Решите систему уравнений A * [u, v] = b
        - Обработайте возможное исключение np.linalg.LinAlgError
    """
    h, w = Ix.shape
    cy, cx = h // 2, w // 2
    half = window_size // 2

    Ix_win = Ix[cy - half:cy + half + 1, cx - half:cx + half + 1]
    Iy_win = Iy[cy - half:cy + half + 1, cx - half:cx + half + 1]
    It_win = It[cy - half:cy + half + 1, cx - half:cx + half + 1]

    A11 = np.sum(Ix_win * Ix_win)
    A12 = np.sum(Ix_win * Iy_win)
    A22 = np.sum(Iy_win * Iy_win)
    A = np.array([[A11, A12], [A12, A22]], dtype=np.float64)

    if np.min(np.linalg.eigvalsh(A)) < 1e-4:
        return None, None

    b = np.array([
        -np.sum(Ix_win * It_win),
        -np.sum(Iy_win * It_win)
    ], dtype=np.float64)

    try:
        flow = np.linalg.solve(A, b)
        return float(flow[0]), float(flow[1])
    except np.linalg.LinAlgError:
        return None, None


def compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size=5):
    """
    Вычисляет оптический поток для патча изображения.

    Аргументы:
        prev_patch: Патч из предыдущего кадра
        curr_patch: Соответствующий патч из текущего кадра
        window_size: Размер окна для LK

    Возвращает:
        Кортеж (u, v) компонентов вектора потока для центра патча

    Примечание:
        - Вычислите пространственные градиенты prev_patch с помощью compute_image_gradients
        - Вычислите временной градиент как разность патчей: It = curr_patch - prev_patch
        - Используйте функцию compute_lk_optical_flow_point для вычисления вектора потока
    """
    prev_patch = prev_patch.astype(np.float64)
    curr_patch = curr_patch.astype(np.float64)
    Ix, Iy = compute_image_gradients(prev_patch)
    It = curr_patch - prev_patch
    return compute_lk_optical_flow_point(Ix, Iy, It, window_size)


def track_point_with_pyramid_lk(prev_pyramid, curr_pyramid, point, window_size=15, max_iterations=10, epsilon=0.01):
    """
    Отслеживает точку между кадрами с использованием пирамидального LK.

    Аргументы:
        prev_pyramid: Пирамида предыдущего кадра (список изображений)
        curr_pyramid: Пирамида текущего кадра (список изображений)
        point: Координаты отслеживаемой точки (x, y) на исходном изображении
        window_size: Размер окна для LK
        max_iterations: Максимальное количество итераций для уточнения каждого уровня
        epsilon: Порог для остановки итераций

    Возвращает:
        Кортеж (new_x, new_y) - новые координаты точки на текущем кадре
        или None если отслеживание неуспешно

    Примечание:
        - Начните обработку с верхнего уровня пирамиды (самое маленькое изображение)
        - Масштабируйте исходную точку для соответствия размеру изображения верхнего уровня
        - Для каждого уровня пирамиды (от верхнего к нижнему):
            1. Масштабируйте общее смещение в 2 раза при переходе на уровень ниже
            2. Пересчитайте позицию точки с учетом масштаба текущего уровня
            3. Примените итеративное уточнение позиции с помощью LK:
                a. Проверьте, что точка и окно вокруг неё находятся в границах изображения
                b. Извлеките патчи из предыдущего и текущего кадров
                c. Вычислите смещение с помощью compute_lk_optical_flow_for_patch
                d. Обновите позицию точки
                e. Остановите итерации, если смещение меньше epsilon
            4. Обновите общее смещение
        - Вычислите финальную позицию точки на исходном изображении
    """
    x0, y0 = float(point[0]), float(point[1])
    num_levels = len(prev_pyramid)
    scale_factor = 0.5
    half = window_size // 2

    u, v = 0.0, 0.0

    for level in range(num_levels - 1, -1, -1):
        scale = scale_factor ** level
        px = x0 * scale
        py = y0 * scale

        if level < num_levels - 1:
            u *= 2.0
            v *= 2.0

        prev_img = prev_pyramid[level]
        curr_img = curr_pyramid[level]
        h, w = prev_img.shape[:2]

        for _ in range(max_iterations):
            x_prev = int(round(px))
            y_prev = int(round(py))
            x_curr = int(round(px + u))
            y_curr = int(round(py + v))

            if (x_prev - half < 0 or x_prev + half >= w or
                    y_prev - half < 0 or y_prev + half >= h or
                    x_curr - half < 0 or x_curr + half >= w or
                    y_curr - half < 0 or y_curr + half >= h):
                return None

            prev_patch = prev_img[y_prev - half:y_prev + half + 1, x_prev - half:x_prev + half + 1]
            curr_patch = curr_img[y_curr - half:y_curr + half + 1, x_curr - half:x_curr + half + 1]

            du, dv = compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size)
            if du is None:
                return None

            u += du
            v += dv

            if abs(du) < epsilon and abs(dv) < epsilon:
                break

    return x0 + u, y0 + v


def lucas_kanade_optical_flow(prev_frame, curr_frame, points,
                             window_size=15, num_pyramid_levels=3,
                             max_iterations=10, epsilon=0.01):
    """
    Вычисляет разреженный оптический поток методом Лукаса-Канаде.

    Аргументы:
        prev_frame: Предыдущий кадр (может быть цветным)
        curr_frame: Текущий кадр (может быть цветным)
        points: Список точек для отслеживания в формате [[x1, y1], [x2, y2], ...]
        window_size: Размер окна для LK
        num_pyramid_levels: Количество уровней в пирамиде изображений
        max_iterations: Максимальное количество итераций на каждом уровне
        epsilon: Порог сходимости для итераций

    Возвращает:
        Кортеж (new_points, status), где:
        - new_points: Массив новых позиций точек в формате [[x1, y1], [x2, y2], ...]
        - status: Массив статусов отслеживания (1 - успешно, 0 - неуспешно)

    Примечание:
        - Преобразуйте входные кадры в полутоновые, если они цветные
        - Нормализуйте изображения к диапазону [0, 1]
        - Создайте пирамиды изображений для обоих кадров
        - Для каждой точки из списка:
            1. Отследите её с помощью track_point_with_pyramid_lk
            2. Сохраните результат в new_points и отметьте статус в status
        - Если отслеживание точки не удалось, установите статус 0 и сохраните исходную точку
    """
    if len(prev_frame.shape) == 3:
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    else:
        prev_gray = prev_frame.copy()

    if len(curr_frame.shape) == 3:
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    else:
        curr_gray = curr_frame.copy()

    prev_gray = prev_gray.astype(np.float64) / 255.0
    curr_gray = curr_gray.astype(np.float64) / 255.0

    prev_pyramid = build_image_pyramid(prev_gray, num_pyramid_levels)
    curr_pyramid = build_image_pyramid(curr_gray, num_pyramid_levels)

    points = np.asarray(points, dtype=np.float64)
    new_points = np.zeros_like(points)
    status = np.zeros(len(points), dtype=np.int32)

    for i, pt in enumerate(points):
        result = track_point_with_pyramid_lk(
            prev_pyramid, curr_pyramid, pt,
            window_size=window_size,
            max_iterations=max_iterations,
            epsilon=epsilon
        )
        if result is None:
            new_points[i] = pt
            status[i] = 0
        else:
            new_points[i] = result
            status[i] = 1

    return new_points, status


def demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    p0 = p0.reshape(-1, 2)  # Преобразуем в формат [[x1, y1], [x2, y2], ...]

    # Сохраняем изначальные точки для отслеживания через все видео
    initial_points = p0.copy()

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (len(p0), 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью нашей реализации
        p1, st = lucas_kanade_optical_flow(
            old_gray,
            frame_gray,
            p0,
            window_size=15,
            num_pyramid_levels=3
        )

        # Выбираем хорошие точки
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        # Рисуем треки
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0[st == 1] = good_new

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [3]:
result_path = demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4')

100%|██████████| 913/913 [00:36<00:00, 24.87it/s]

Результат сохранен в output_my_LK.mp4


### Релизация OpenCV - cv2.calcOpticalFlowPyrLK

In [4]:
def demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео с использованием cv2.calcOpticalFlowPyrLK.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Параметры для Lucas-Kanade оптического потока
    lk_params = dict(
        winSize=(15, 15),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (100, 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью встроенной функции cv2.calcOpticalFlowPyrLK
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        # Выбираем хорошие точки
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

        # Рисуем треки
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0 = good_new.reshape(-1, 1, 2)

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [5]:
result_path = demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_LK.mp4')

100%|██████████| 913/913 [00:03<00:00, 246.62it/s]

Результат сохранен в output_opencv_LK.mp4


### Задание 2

В базовой реализации у кода есть одна важная проблема - ключевые точки инициализируются единожды. В реальных задачах необходимо отслеживать точки, которые исчезают из кадра и появляются в других местах. Реализуйте механизм, который будет отслеживать точки, которые пропадают из кадра и добавлять новые точки в те места, где они появляются. Для этого вам нужно будет реализовать механизм поиска новых точек на изображении.

In [6]:
def demo_optical_flow_with_refresh(
    video_path='data/slow_traffic_small.mp4',
    output_path='output_my_LK_refresh.mp4',
    max_corners=100,
    min_distance=7,
    quality_level=0.3,
    block_size=7,
    window_size=15,
    num_pyramid_levels=3,
):
    """
    Трекинг LK с подстановкой новых точек: потерянные удаляются,
    новые Shi-Tomasi добавляются в свободные области кадра.
    """
    cap = cv2.VideoCapture(video_path)
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    feature_params = dict(
        maxCorners=max_corners,
        qualityLevel=quality_level,
        minDistance=min_distance,
        blockSize=block_size,
    )

    ret, old_frame = cap.read()
    if not ret:
        cap.release()
        out.release()
        return None

    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    p0 = p0.reshape(-1, 2)

    mask = np.zeros_like(old_frame)
    colors = np.random.randint(0, 255, (max_corners, 3))
    point_colors = colors[:len(p0)].copy()

    for _ in tqdm(range(length - 1)):
        ret, frame = cap.read()
        if not ret:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        p1, st = lucas_kanade_optical_flow(
            old_gray, frame_gray, p0,
            window_size=window_size,
            num_pyramid_levels=num_pyramid_levels,
        )

        good_mask = st == 1
        good_new = p1[good_mask]
        good_old = p0[good_mask]
        good_colors = point_colors[good_mask]

        for new, old, col in zip(good_new, good_old, good_colors):
            a, b = new
            c, d = old
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), col.tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, col.tolist(), -1)

        img = cv2.add(frame, mask)
        out.write(img)

        # Маска: запрещаем искать углы рядом с уже отслеживаемыми точками
        feature_mask = np.full(frame_gray.shape, 255, dtype=np.uint8)
        for pt in good_new:
            cv2.circle(feature_mask, (int(pt[0]), int(pt[1])), min_distance, 0, -1)

        n_needed = max_corners - len(good_new)
        if n_needed > 0:
            new_pts = cv2.goodFeaturesToTrack(
                frame_gray,
                mask=feature_mask,
                maxCorners=n_needed,
                qualityLevel=quality_level,
                minDistance=min_distance,
                blockSize=block_size,
            )
            if new_pts is not None:
                new_pts = new_pts.reshape(-1, 2)
                new_colors = np.random.randint(0, 255, (len(new_pts), 3))
                if len(good_new) > 0:
                    p0 = np.vstack([good_new, new_pts])
                    point_colors = np.vstack([good_colors, new_colors])
                else:
                    p0 = new_pts
                    point_colors = new_colors
            else:
                p0 = good_new
                point_colors = good_colors
        else:
            p0 = good_new
            point_colors = good_colors

        old_gray = frame_gray.copy()

    cap.release()
    out.release()
    print(f"Результат сохранен в {output_path}")
    return output_path


result_path = demo_optical_flow_with_refresh(
    video_path='data/slow_traffic_small.mp4',
    output_path='output_my_LK_refresh.mp4',
)

100%|██████████| 913/913 [03:52<00:00,  3.92it/s]

Результат сохранен в output_my_LK_refresh.mp4


### Вопрос 4

В чем основное отличие разреженного (sparse) оптического потока Lucas-Kanade от плотного (dense) оптического потока (например, метода Farneback)?

**Ответ:**

**Lucas–Kanade (sparse):** поток оценивается только в **заранее выбранных** хорошо отслеживаемых точках (углы Shi-Tomasi и т.п.). Локальная модель в окне, быстро для небольшого числа точек, результат — набор векторов $(u, v)$ в отдельных пикселях.

**Farneback (dense):** поток вычисляется **в каждом пикселе** изображения за счёт полиномиальной аппроксимации окрестности и предположения о гладкости поля на больших областях. Даёт полное поле скоростей, но дороже по вычислениям.

Итого: sparse — «где есть текстура и углы», dense — «везде на кадре».


## Farneback (dense)

Метод Farneback носит несколько более глобальный характер, чем метод Лукаса-Канаде. Он опирается на предположение о том, что на всем изображении оптический поток будет достаточно гладким.

# Вопрос 5

Перечислите основные шаги алгоритма Farneback для расчета оптического потока.

**Ответ:**

1. **Построение пирамиды** изображений для двух кадров (многоуровневое уточнение).
2. **Полиномиальное разложение** интенсивности в локальном окне каждого пикселя (коэффициенты аппроксимации $f(x,y)$ и их производные).
3. **Связь коэффициентов** между кадрами при смещении $(u, v)$ — из разности полиномов получают уравнения для сдвига.
4. **Оценка смещения** в окне с учётом сглаживания (взвешивание по Гауссу, параметр $\sigma$).
5. **Итеративное уточнение** на каждом уровне пирамиды.
6. **Масштабирование и перенос** оценки на более детальный уровень, повтор до полного разрешения.
7. **Формирование плотного поля** $(u, v)$ для всех пикселей.


### Вопрос 6

Каким образом в методе Farneback обрабатываются большие смещения объектов между кадрами?

**Ответ:**

Как и в пирамидальном LK, используется **пирамида изображений** (параметры `pyr_scale`, `levels` в `cv2.calcOpticalFlowFarneback`). На грубом уровне большое смещение выглядит как малое в пикселях; оценка потока там устойчивее. При переходе на более детальный уровень вектор **масштабируется** и **уточняется** итерациями. Дополнительно помогает **полиномиальная модель** окрестности и **сглаживание** поля потока, что стабилизирует оценку при умеренных нелинейностях движения.


In [7]:
def demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_Farneback.mp4'):
    """
    Демонстрация работы алгоритма плотного оптического потока Farneback на видео.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Берем первый кадр и преобразуем его в оттенки серого
    ret, frame1 = cap.read()
    if not ret:
        print('Не удалось прочитать видео')
        return None

    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    # Создаем HSV-изображение для визуализации потока
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255  # Насыщенность устанавливаем на максимум

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame2 = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток методом Farneback
        # Параметры:
        # - 0.5: коэффициент масштабирования для пирамиды изображений
        # - 3: кол-во уровней пирамиды
        # - 15: размер окна для усреднения
        # - 3: число итераций на каждом уровне пирамиды
        # - 5: размер окна для полиномиальной аппроксимации
        # - 1.2: стандартное отклонение для сглаживания
        flow = cv2.calcOpticalFlowFarneback(
            prvs, next_frame, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        # Преобразуем векторы потока из декартовых координат в полярные
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Кодируем направление потока как оттенок (hue)
        hsv[..., 0] = ang * 180 / np.pi / 2

        # Кодируем величину потока как яркость (value)
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        # Преобразуем HSV в BGR для отображения
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        # Записываем результат
        out.write(bgr)

        # Обновляем предыдущий кадр
        prvs = next_frame

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [8]:
result_path = demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_farneback.mp4')

100%|██████████| 913/913 [00:46<00:00, 19.44it/s]

Результат сохранен в output_opencv_farneback.mp4


### Вопрос 7

Как влияет предварительная обработка изображений (фильтрация шума, выравнивание гистограмм) на качество оптического потока, получаемого методом Farneback? Предложите оптимальный пайплайн предобработки.

**Ответ:**

**Фильтрация шума** (лёгкий Gaussian или bilateral) обычно **улучшает** поток: меньше ложных градиентов от шума, стабильнее полиномиальная аппроксимация. Слишком сильное размытие **ухудшает** результат — теряются мелкие структуры, смещения «размазываются».

**Выравнивание гистограмм** (глобальное HE, агрессивный CLAHE) часто **вредит**, потому что нарушает **постоянство яркости** между кадрами: один и тот же объект может получить разную яркость на $t$ и $t+1$. Локальный CLAHE иногда помогает на тёмных сценах, но его нужно применять **одинаково** к обоим кадрам и умеренно.

**Оптимальный пайплайн:**

1. Конвертация в grayscale (если цветное видео).
2. Опционально: лёгкое **bilateral** или Gaussian ($\sigma \approx 0.5$–$1.0$) для подавления шума с сохранением границ.
3. **Не** делать независимое выравнивание гистограмм по кадрам; при необходимости — одна и та же глобальная нормализация (например, фиксированный gain/offset по экспозиции камеры).
4. Одинаковый тип `uint8` / `float32` и диапазон для пары кадров перед `calcOpticalFlowFarneback`.
5. При сильном шуме — чуть увеличить `poly_sigma` и размер окна Farneback вместо агрессивной предобработки.
